In [ ]:
pip install -U langgraph langchain langchain-core wikipedia-api pydantic ipython

In [ ]:
pip install -U langchain-groq

In [ ]:
pip install wikipedia-api

In [34]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv(override=True)  
assert os.getenv("GROQ_API_KEY"), "API_KEY not found."

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1
)


In [38]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph import MessagesState
import operator
from typing import Annotated

from langgraph.graph import add_messages
from langchain_core.messages import SystemMessage

In [36]:
class ResearcherState(TypedDict):
    task: str
    research: str
    messages: Annotated[list, add_messages]
    trace: Annotated[list, operator.add]
    

    

In [40]:
#Resarcher Graph

from langgraph.prebuilt import ToolNode
import wikipediaapi
from langchain_core.tools import tool

wiki_wiki = wikipediaapi.Wikipedia(
    user_agent="LangGraphAssignment/1.0 (student@example.com)",
    language="en"
)


@tool


def wikipedia_search(query: str):
    
    """Search Wikipedia and return raw page content for the given topic."""
    
    results = wiki_wiki.search(query, limit=3)
    research = []
    
    for title, page in results.pages.items():
        research.append(f"# {title}\n{page.text[:3000]}")

    return "\n\n".join(research)


tools = [wikipedia_search]
llm_with_tools = llm.bind_tools(tools)


from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

def Researcher(state):
    messages = state["messages"]

    if not messages:
        messages = [HumanMessage(content=state["task"])]

    response = llm_with_tools.invoke([
        SystemMessage(content="""
        You are a researcher.
        Research the given task using Wikipedia.
        Collect raw information only.
        Do not summarize or format the research.
        """),
        *messages
    ])

    research = "\n\n".join(
        msg.content for msg in messages
        if isinstance(msg, ToolMessage) # pyright: ignore[reportArgumentType]
    ) # type: ignore

    return {
        "research": research,
        "messages": [response],
        "trace": ["researcher"]
    }

def route(state: ResearcherState):
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END    
    
builder=StateGraph(ResearcherState)
builder.add_node("researcher", Researcher)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "researcher")
builder.add_conditional_edges(
    "researcher",
    route,
    {
        "tools": "tools",
        END: END
    }
)

builder.add_edge("tools", "researcher")


graph=builder.compile()


In [ ]:
#Test researcher graph
researcher_graph = builder.compile()

print("RESEARCH:")
print(result["research"])

print("\nTRACE:")
print(result["trace"])

AttributeError: 'function' object has no attribute 'invoke'

In [ ]:
def Writer(state):
    messages = f"Writing based on research: {state['research']}"
    state['messages'] = messages
    return state

def Reviewer(state):
    messages = f"Reviewing the messages: {state['messages']}"
    state['messages'] = messages
    return state

In [ ]:
builder=StateGraph(State)
builder.add_node("Researcher", Researcher)
builder.add_node("Writer", Writer)    
builder.add_node("Reviewer", Reviewer)



builder.addedge(START, Researcher)
builder.addedge(Researcher, Writer) 
builder.addedge(Writer, Reviewer)
builder.addedge(Reviewer, END)

graph=builder.compile()


AttributeError: 'StateGraph' object has no attribute 'addedge'